In [30]:
import os
import base64
import requests
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 1. Grab your API Key
api_key = os.environ.get("ROBOFLOW_API_KEY", "input_api_key")

def predict_with_full_confidence(image_path, project_name, version):
    # Display the image inline
    img = mpimg.imread(image_path)
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    # 2. Encode the image into a base64 string so we can send it over the internet
    with open(image_path, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode("ascii")

    # 3. Build the direct API URL
    # Notice we are forcing the confidence down to 0.01 (1%) right in the URL
    url = f"https://classify.roboflow.com/{project_name}/{version}"
    params = {
        "api_key": api_key,
        "confidence": "0.01"
    }

    # 4. Fire the request directly to the server
    print(f"--- Full Confidence Distribution ---")
    response = requests.post(url, params=params, data=encoded_image, headers={"Content-Type": "application/x-www-form-urlencoded"})

    # Catch any connection errors
    if response.status_code != 200:
        print(f"API Error: {response.text}")
        return

    # 5. Parse the returned JSON
    data = response.json()
    predictions = data.get("predictions", [])

    # --- NEW LOGIC: Enforce a standard 3-class output ---

    # 1. Create a dictionary with your baseline classes set to 0%
    # IMPORTANT: Ensure these strings perfectly match your Roboflow class names
    final_scores = {
        "young_seal": 0.0,
        "Adult Male": 0.0,
        "Adult Female": 0.0
    }

    # 2. Loop through the API results. If the API found it, overwrite the 0.0
    for pred in predictions:
        seal_class = pred["class"]
        # This prevents crashes if Roboflow returns a class you didn't expect
        if seal_class in final_scores:
            final_scores[seal_class] = pred["confidence"]
        else:
            final_scores[seal_class] = pred["confidence"]

    # 3. Print the guaranteed output
    for seal_class, confidence in final_scores.items():
        percentage = confidence * 100
        print(f"{seal_class}: {percentage:.0f}%")

    print("-" * 34)

# --- Run the Test ---
# Use the exact project ID from your screenshot
predict_with_full_confidence(
    image_path="Add Imput Image Path",
    project_name="individual-classification",
    version="1"
)

young_seal: 99%
Adult Male: 0%
Adult Female: 1%
----------------------------------
